# NextStudy Notebook

In diesem Notebook zeige ich die wichtigsten Teile von NextStudy noch einmal Schritt für Schritt. Ich benutze dabei die gleiche `index.py`, die auch als Programm abgegeben wird. Gestartet werden kann das Programm direkt mit Python oder per Doppelklick über `start_nextstudy_windows.bat` beziehungsweise `start_nextstudy_macos.command`.

## 1. Programmcode laden

Zuerst lade ich die Python-Datei. Dadurch kann ich im Notebook die Klassen und Funktionen benutzen, ohne den Code noch einmal komplett neu zu schreiben.

In [2]:
from pathlib import Path
import importlib.util
import json

module_path = Path("index.py")
if not module_path.exists():
    module_path = Path("new/index.py")

spec = importlib.util.spec_from_file_location("nextstudy", module_path)
nextstudy = importlib.util.module_from_spec(spec)
spec.loader.exec_module(nextstudy)

print(f"Modul geladen: {module_path.name}")

Modul geladen: index.py


## 2. Beispielthemen erstellen

Für die Demo erstelle ich drei Themen. Wichtig ist hier vor allem die Schwierigkeit, weil daraus später die Gewichtung entsteht: leicht = 1, mittel = 2 und schwer = 3.

In [3]:
themen = [
    nextstudy.Thema("Ableitungen", "schwer"),
    nextstudy.Thema("Gleichungen", "mittel"),
    nextstudy.Thema("Formeln wiederholen", "leicht"),
]

for thema in themen:
    print(f"{thema.name}: {thema.schwierigkeit}, Gewichtung {thema.gewichtung}")

Ableitungen: schwer, Gewichtung 3
Gleichungen: mittel, Gewichtung 2
Formeln wiederholen: leicht, Gewichtung 1


## 3. Lernplan erzeugen

Jetzt wird aus den Themen ein Plan erstellt. Die Funktion `lernplan_erstellen()` sortiert die Themen nach Gewichtung. Schwere Themen bekommen dadurch mehr Platz im Plan. Der letzte Tag ist für Wiederholung gedacht und bei längeren Plänen wird auch Prüfungsvorbereitung eingeplant.

In [6]:
fach = "Mathe"
tage = 5
lernzeit = 45
plan = nextstudy.lernplan_erstellen(themen, tage, lernzeit)

for einheit in plan:
    print(f"Tag {einheit.tag}: {einheit.thema} - {einheit.aufgabe}")

Tag 1: Ableitungen - Ableitungen intensiv lernen, Aufgaben üben und Fehler notieren
Tag 2: Ableitungen - Ableitungen intensiv lernen, Aufgaben üben und Fehler notieren
Tag 3: Ableitungen - Ableitungen intensiv lernen, Aufgaben üben und Fehler notieren
Tag 4: Prüfungsvorbereitung - Schwächen wiederholen, Zusammenfassung lesen und Beispielaufgaben lösen
Tag 5: Wiederholung - Alle Themen wiederholen, offene Fragen klären und Mini-Selbsttest machen


## 4. Aufgabe abhaken und Statistik ansehen

Eine Aufgabe ist am Anfang immer `offen`. Wenn sie erledigt wurde, setzt `erledigen()` den Status auf `abgeschlossen`. Danach kann man gut sehen, wie sich die Statistik verändert.

In [7]:
plan[0].erledigen()
nextstudy.statistik_anzeigen(plan)


===== Statistik =====
Aufgaben gesamt:    5
Abgeschlossen:      1
Offen:              4
Lernzeit gesamt:    225 Minuten
Erledigte Lernzeit: 45 Minuten
Fortschritt:        20 %
Hinweis:            Starte mit der ersten Aufgabe. Danach wird es einfacher.


## 5. Daten für JSON vorbereiten

Zum Speichern müssen die Objekte in normale Dictionaries umgewandelt werden. Genau dafür gibt es in den Klassen die Methode `to_dict()`.

In [8]:
daten = {
    "fach": fach,
    "tage": tage,
    "lernzeit": lernzeit,
    "themen": [thema.to_dict() for thema in themen],
    "plan": [einheit.to_dict() for einheit in plan],
}

auszug = {key: daten[key] for key in ["fach", "tage", "lernzeit", "themen"]}
print(json.dumps(auszug, indent=4, ensure_ascii=False))

{
    "fach": "Mathe",
    "tage": 5,
    "lernzeit": 45,
    "themen": [
        {
            "name": "Ableitungen",
            "schwierigkeit": "schwer",
            "gewichtung": 3
        },
        {
            "name": "Gleichungen",
            "schwierigkeit": "mittel",
            "gewichtung": 2
        },
        {
            "name": "Formeln wiederholen",
            "schwierigkeit": "leicht",
            "gewichtung": 1
        }
    ]
}


## 6. Menülogik mit Dispatch-Tabelle

Im Hauptprogramm wird die Menüauswahl über das Dictionary `aktionen` gesteuert. Das Dictionary speichert Funktionsreferenzen. Die Funktion wird also nicht sofort ausgeführt, sondern erst dann, wenn die passende Eingabe ausgewählt wurde. Der aktuelle Zustand liegt in `daten`.

In [ ]:
daten = {"fach": fach, "themen": themen, "tage": tage, "lernzeit": lernzeit, "plan": plan}

aktionen = {
    "2": lambda: nextstudy.plan_anzeigen(daten["fach"], daten["plan"]),
    "4": lambda: nextstudy.statistik_anzeigen(daten["plan"]),
}

aktion = aktionen.get("4")
if aktion:
    aktion()

## 7. Sicherheits-Backup

Damit das Programm nicht direkt abstürzt, werden kritische Stellen mit `try/except` abgesichert. Wenn eine Aktion einen Fehler auslöst, wird eine Meldung angezeigt und danach kann das Menü weiterlaufen.

In [ ]:
try:
    aktion()
except Exception as fehler:
    print(f"Unerwarteter Fehler: {fehler}")
    print("Das Sicherheits-Backup hat den Absturz verhindert.")

## 8. Ergebnis

Das Notebook zeigt die Grundidee des Programms: Themen werden als Objekte gespeichert, daraus entsteht ein gewichteter Lernplan, erledigte Aufgaben verändern die Statistik, der Programmzustand liegt in `daten`, die Menüauswahl kann über eine Dispatch-Tabelle laufen, Fehler werden durch ein Sicherheits-Backup abgefangen und das Programm kann über Startdateien direkt geöffnet werden.